# Connect to an Azure AI Foundry Agent

This notebook walks through connecting to a Foundry agent and sending it a prompt.

You will:
1. Install dependencies
2. Configure endpoint and agent settings
3. Create authenticated clients
4. Send a request to the agent
5. Print the response

## Step 1: Install required packages

Run this once per environment. If packages are already installed, you can skip this cell.

Before running the client creation step locally, sign in to Azure CLI so `DefaultAzureCredential` can authenticate:

`az login`

If you are using managed identity or service principal environment variables, this login step may not be needed.

In [12]:
%pip install --pre "azure-ai-projects>=2.0.0b1" azure-identity

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Step 2: Import SDKs

These imports provide Azure authentication and access to your Foundry project.

In [13]:
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient

## Step 3: Configure your Foundry project and target agent

Update values if your endpoint, agent name, or version changes.

In [18]:
myEndpoint =  "https://ckriutzfield-9041-resource.services.ai.azure.com/api/projects/ckriutzfield-9041"
myAgent = "zava-news-agent"
myVersion = "5"

user_prompt = "Can you list out the suggestions for inventory changes?"

## Step 4: Create authenticated clients

`DefaultAzureCredential` uses your active sign-in context (for example, Azure CLI login in local dev).

In [19]:
project_client = AIProjectClient(
    endpoint=myEndpoint,
    credential=DefaultAzureCredential(),
)

openai_client = project_client.get_openai_client()
print("✅ Clients created")

✅ Clients created


## Step 5: Ask the agent a question

This calls the Responses API with an `agent_reference` payload.

In [20]:
from IPython.display import display, Markdown

response = openai_client.responses.create(
    input=[{"role": "user", "content": user_prompt}],
    extra_body={
        "agent_reference": {
            "type": "agent_reference",
            "name": myAgent,
            "version": myVersion,
        }
    },
)
text = response.output_text
display(Markdown(text))

[
{"productSKU":"ZCA-SC-PRO","IsIncrease":true,"percentage":50,"reason":"Smart Cleats Pro demand is accelerating and close to sell-out (West region e‑commerce spike and large B2B bulk orders); increase stock to cover backorders and corporate requests  "},
{"productSKU":"ZCF-PR-LEG","IsIncrease":true,"percentage":35,"reason":"Field Premium Performance Leggings are top sellers with high revenue and reported backorders; increase inventory to relieve stockouts and capture momentum  "},
{"productSKU":"ZCF-EL-SST","IsIncrease":true,"percentage":40,"reason":"Field Elite Short Sleeve Top is backordered and seeing a Direct eCommerce surge; raise stock and expedite replenishment to prevent lost sales  "},
{"productSKU":"ZCS-JRS-CUS","IsIncrease":false,"percentage":30,"reason":"Systems Custom Team Jersey has concentrated exposure to Fabrikam and East region deliveries are suspended / Fabrikam under investigation; reduce on-hand inventory or reallocate until customer/region risk is clarified  "},
{"productSKU":"ZCR-FBR-FLD","IsIncrease":false,"percentage":25,"reason":"ZavaCore Fiber — Field Grade is heavily purchased by East customers (including Fabrikam) and is impacted by East region disruptions and customer credit risk; lower committed stock to avoid spoilage/overhang and reallocate to West demand where appropriate  "},
{"productSKU":"ZCR-FBR-SYS","IsIncrease":true,"percentage":20,"reason":"ZavaCore Fiber — Systems Grade is high-value, in allocation with longer lead times, and West region B2B demand is rising; increase procurement to build a buffer for production and R&D needs  "}
]